# 🎵 Sound Metadata Extractor — Colab Launcher

> **Code source:** [`ahmedHosney600/Sounds-Meta-Info-Extractor`](https://github.com/ahmedHosney600/Sounds-Meta-Info-Extractor)  
> **Data source:** Your Google Drive sounds folder  
> **Outputs:** Saved back to Google Drive

---
### Run order
1. **Cell 1** — Mount Google Drive (for sounds + outputs)
2. **Cell 2** — Clone project from GitHub & install dependencies
3. **Cell 3** — ⚙️ **Set your configuration** (edit only this cell)
4. **Cell 4** — ▶️ Run the extractor
5. **Cell 5** — (Optional) Preview results table

In [ ]:
# ============================================================
# CELL 1 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

In [ ]:
# ============================================================
# CELL 2 — Clone from GitHub & Install Dependencies
# ============================================================
import os, shutil

REPO_URL  = 'https://github.com/ahmedHosney600/Sounds-Meta-Info-Extractor.git'
CLONE_DIR = '/content/Sounds-Meta-Info-Extractor'

# Always pull the latest version from GitHub
if os.path.exists(CLONE_DIR):
    print('Repo already cloned — pulling latest changes...')
    !git -C "{CLONE_DIR}" pull
else:
    print('Cloning repository...')
    !git clone "{REPO_URL}" "{CLONE_DIR}"

# System dependency: ffmpeg for AIFF, M4A, OGG, OPUS, etc.
print('Installing ffmpeg...')
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ ffmpeg installed')

# Python dependencies
print('Installing Python libraries...')
!pip install -q -r "{CLONE_DIR}/requirements.txt"
print('✅ All libraries installed')

# Add project to Python path
import sys
if CLONE_DIR not in sys.path:
    sys.path.insert(0, CLONE_DIR)
print(f'✅ Project added to sys.path: {CLONE_DIR}')

In [ ]:
# ============================================================
# CELL 3 — ⚙️ CONFIGURATION  (only edit this cell)
# ============================================================
import os

# ── Sounds folder on Google Drive ──────────────────────────────────
# Full path to your sounds folder as mounted in Colab.
# Example: '/content/drive/MyDrive/Sound Libraries/BLOW'
os.environ['SOUNDS_FOLDER'] = '/content/drive/MyDrive/SOUNDS'

# ── Google Drive Folder ID (for preview/download links) ─────────
# Open your sounds folder in Drive in the browser.
# The URL will be: https://drive.google.com/drive/folders/<ID>
# Paste the ID below. Leave empty to skip link generation.
os.environ['DRIVE_FOLDER_ID'] = ''  # e.g. '1AbCdEfGhIjKlMnOpQr'

# ── Output folder (saved to your Google Drive) ────────────────
os.environ['OUTPUT_FOLDER'] = '/content/drive/MyDrive/sounds_metadata_output'

# ── Scan settings ───────────────────────────────────────────
os.environ['RECURSIVE']        = 'true'   # search subfolders?
os.environ['MAX_FILE_SIZE_MB'] = '1000'    # skip files > this MB (0 = no limit)

print('✅ Configuration:')
print(f'   SOUNDS_FOLDER   = {os.environ["SOUNDS_FOLDER"]}')
print(f'   DRIVE_FOLDER_ID = {os.environ["DRIVE_FOLDER_ID"] or "(not set — Drive links disabled)"}')
print(f'   OUTPUT_FOLDER   = {os.environ["OUTPUT_FOLDER"]}')
print(f'   RECURSIVE       = {os.environ["RECURSIVE"]}')

In [ ]:
# ============================================================
# CELL 4 — ▶️ Run the Extractor
# ============================================================
CLONE_DIR = '/content/Sounds-Meta-Info-Extractor'

# Run main.py from the cloned repo.
# Environment variables set in Cell 3 are automatically inherited.
!cd "{CLONE_DIR}" && python main.py

In [ ]:
# ============================================================
# CELL 5 — (Optional) Preview Results Table
# ============================================================
import json, os
import pandas as pd
from IPython.display import display

output_folder = os.environ.get('OUTPUT_FOLDER', '/content/drive/MyDrive/sounds_metadata_output')
json_path = f'{output_folder}/sounds.json'

with open(json_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f'Loaded {len(data)} records from {json_path}')

PREVIEW_COLS = [
    'filename', 'extension', 'sf_duration_seconds', 'sf_sample_rate',
    'sf_bit_depth', 'sf_channel_label', 'lb_tempo_bpm',
    'ai_top_class', 'ai_top_score', 'heuristic_sound_type',
    'fn_parsed_category', 'fn_parsed_description',
    'drive_preview_url',
]

def flatten(r):
    return {k: (str(v) if isinstance(v, (list, dict)) else v) for k, v in r.items()}

df = pd.DataFrame([flatten(r) for r in data])
available = [c for c in PREVIEW_COLS if c in df.columns]
display(df[available].head(20))